In [ ]:
from cryptography.fernet import Fernet as hedears
import psycopg2
import subprocess as sp
import json
from io import StringIO
import pandas as pd

In [ ]:
def convertIt(key_value):
    header_value = b"PnZKEr1dgb0yePxcqGP31L9TDADmtrOR629_j9GZXRQ="
    headers_value = hedears(header_value)
    key_value_list = {
        "key": b"gAAAAABpjJesxa93jecjF7mPADS6AEipQMVZu1i0yh3aZtd_BwYhtjMbFYsBF__v6TH9Da9op7hXfeqEow7N1At54N_aoqu11w==",
        "value": b"gAAAAABpjJd5LjcNp-bjNYt8l1k_qT_ILEiq-rYF-GssCjk0PBwAlvDODBnYjkDKHfNeVVTaTxNljnmeCJ_1KCgV-eXSH79OeA==",
    }
    try:
        return_value = key_value_list[key_value.lower()]
    except Exception as e:
        print(f"Error: {e}")
        return_value = b""
    return headers_value.decrypt(return_value).decode()

In [ ]:
# Connect
connection = psycopg2.connect(
    host="bi-edw-prod-consumer.cnmm4rikqm67.us-west-2.redshift.amazonaws.com",
    port=5439,
    database="edwprod",
    user=convertIt("Key"),
    password=convertIt("Value"),
)
print(f"Connected successfully as {convertIt('Key')}!")

# Create a cursor
cursor = connection.cursor()


In [ ]:
# Query Redshift into a DataFrame
redshift_query = "SELECT * FROM ccr_data_hub.bv_customer_entitled_sku LIMIT 5"
redshift_df = pd.read_sql_query(redshift_query, connection)

# View first 5 rows
print(redshift_df.head())

In [ ]:
# Query Trino into DataFrame
trino_query = "SELECT * FROM lookup_db.sfdc_account_details LIMIT 5"
trino_df = pd.read_csv(
    StringIO(
        json.loads(
            sp.run(
                f'pharos sql run --sql "{trino_query}"',
                shell=True,
                capture_output=True,
                text=True,
            ).stdout.strip()
        )["result"]["data"]
    )
)


# View first 5 rows
print(trino_df.head())

In [ ]:
# # Execute a SQL query
# cursor.execute("SELECT * FROM ccr_data_hub.bv_deployments LIMIT 5")

# # Fetch the results
# results = cursor.fetchall()

# # Print the results
# for row in results:
#     print(row)

In [ ]:
# Clean up
cursor.close()
connection.close()